[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/37_dpo_loss.ipynb)

# 🔴 Hard: DPO Loss

Implement the **Direct Preference Optimization** loss — the standard loss for LLM alignment.

$$\mathcal{L}_{\text{DPO}} = -\log \sigma\Big(\beta \big[\log\frac{\pi(y_w)}{\text{ref}(y_w)} - \log\frac{\pi(y_l)}{\text{ref}(y_l)}\big]\Big)$$

### Signature
```python
def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1) -> Tensor:
    # All inputs: (B,) log-probabilities
    # Returns: scalar loss
```

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [3]:
import torch
import torch.nn.functional as F

In [17]:
def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1):
    #DPO loss consists in running the current policy to obtain 2 outcomes. Then a judge picks one and we train the model to maximize the logp of the winner and minimize the one of the loser.
    #We can do this on a batch of couples (winner, loser) and then recollect (if online DPO otherwise no recllection) them with the newly trained model (if batch is too big we will be too much off-policy since our model will be trained on old responses).
    #The 'ref' model is always the same one over all training.
    """
    all (B,)
    """

    return -torch.log(F.sigmoid(beta * (policy_chosen_logps - ref_chosen_logps - policy_rejected_logps + ref_rejected_logps))).mean()

In [18]:
# 🧪 Debug
chosen = torch.tensor([0.0, 0.0])
rejected = torch.tensor([-5.0, -5.0])
ref_c = torch.tensor([-1.0, -1.0])
ref_r = torch.tensor([-1.0, -1.0])
print('Loss:', dpo_loss(chosen, rejected, ref_c, ref_r, beta=0.1).item())

Loss: 0.4740769565105438


In [19]:
# ✅ SUBMIT
from torch_judge import check
check('dpo_loss')


🧪 Testing: DPO (Direct Preference Optimization) Loss (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Easy pair: small loss (3.4ms)
  ✅ [2/5] Hard pair: large loss (0.4ms)
  ✅ [3/5] Gradient flow (0.6ms)
  ✅ [4/5] Mathematical correctness (2.9ms)
  ✅ [5/5] Beta scaling (1.9ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (9.3ms total)
  Progress saved. Run status() to see your dashboard.

